In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

import TutorialFeedback from '@site/src/components/TutorialFeedback';





*Оцінка використання: 4 хвилини на процесорі Heron r2. (ПРИМІТКА: Це лише оцінка. Твій час виконання може відрізнятися.)*

## Результати навчання

Після завершення цього підручника ти дізнаєшся наступне:
* Як реалізувати дальній гейт CNOT з використанням динамічних схем з вимірюваннями в середині схеми (MCM) та класичним прямим зв'язком;
* Як реалізувати еквівалентний гейт за допомогою унітарного підходу на основі SWAP;
* Як порівняти обидва підходи, вимірюючи точність гейта як функцію відстані між кубітами.

## Передумови

Ми рекомендуємо користувачам ознайомитися з такими темами перед проходженням цього підручника:
* [Базові концепції квантових обчислень](/learning/courses/basics-of-quantum-information), включаючи стани Белла, заплутування та квантові гейти;
* Знайомство з [динамічними схемами](/guides/classical-feedforward-and-control-flow) (вимірювання в середині схеми та класичний прямий зв'язок);
* Базові знання [Qiskit SDK](https://docs.quantum.ibm.com/guides) та [Qiskit Runtime](/guides/compute-services#qiskit-runtime), а також доступ до [облікового запису IBM Quantum&reg;](/guides/cloud-setup).

## Загальні відомості

Дальнє заплутування між віддаленими кубітами є складним завданням на пристроях з обмеженою зв'язністю. Цей підручник показує, як динамічні схеми можуть генерувати таке заплутування шляхом реалізації дальнього керованого-X (LRCX) гейта з використанням протоколу на основі вимірювання.

Слідуючи підходу Еліси Боймер та ін. у [1](#ref-1), метод використовує вимірювання в середині схеми та прямий зв'язок для досягнення гейтів постійної глибини незалежно від відстані між кубітами. Він створює проміжні пари Белла, вимірює один кубіт з кожної пари та застосовує класично зумовлені гейти для поширення заплутування через пристрій. Це дозволяє уникнути довгих ланцюжків SWAP, зменшуючи як глибину схеми, так і вплив помилок двокубітних гейтів.

У цьому блокноті ми адаптуємо протокол для обладнання IBM Quantum та порівнюємо його продуктивність як функцію відстані між керуючим та цільовим кубітами, зіставляючи його з унітарним базовим підходом на основі SWAP.

## Вимоги

Перед початком цього підручника переконайся, що у тебе встановлено наступне:

- Qiskit SDK v2.0 або пізніше, з підтримкою [візуалізації](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime v0.37 або пізніше (`pip install qiskit-ibm-runtime`)
- Qiskit Aer v0.17 або пізніше (`pip install qiskit-aer`)

## Налаштування

In [2]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.classical import expr
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.visualization import plot_circuit_layout
from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    Batch,
    SamplerV2 as Sampler,
)
import matplotlib.pyplot as plt
import numpy as np

## Приклад на симуляторі малого масштабу

Перш ніж запускати на реальному QPU, ми перевіряємо, що як динамічна, так і унітарна схеми виробляють ідеальний стан Белла на симуляторі без шуму. Ми використовуємо Qiskit Runtime `Sampler` з `AerSimulator` як режим бекенду, з невеликою відстанню 6.

### Крок 1: Відображення класичних входів на квантову задачу

Тепер ми реалізуємо дальній гейт CNOT між двома віддаленими кубітами, слідуючи конструкції динамічної схеми, показаній нижче (адаптовано з Рис. 1a в Посиланні [1](#ref-1)). Ключова ідея полягає у використанні "шини" допоміжних кубітів, ініціалізованих до $|0\rangle$, для посередництва в дальній телепортації гейта.

![Long-range CNOT circuit](../docs/images/tutorials/long-range-entanglement/dynamic_vs_unitary_long_range_illustration.avif)

Як показано на малюнку, процес працює наступним чином:
1. Підготувати ланцюжок пар Белла, що з'єднують керуючий та цільовий кубіти через проміжні допоміжні кубіти.
2. Виконати вимірювання Белла між незаплутаними сусідніми кубітами, обмінюючи заплутування крок за кроком, доки керуючий та цільовий кубіти не поділять пару Белла.
3. Використовувати цю пару Белла для телепортації гейта, перетворюючи локальний CNOT на детерміністичний дальній CNOT постійної глибини.

Цей підхід замінює довгі ланцюжки SWAP на протокол постійної глибини, зменшуючи вплив помилок двокубітних гейтів і роблячи операцію масштабованою зі розміром пристрою.

У наступному ми спершу розглянемо реалізацію схеми LRCX на основі динамічних схем. Наприкінці ми також надамо унітарну реалізацію для порівняння, щоб підкреслити переваги динамічних схем у цій ситуації.

#### Ініціалізація схеми

Ми починаємо з простої квантової задачі, яка слугуватиме основою для порівняння. Зокрема, ми ініціалізуємо схему з керуючим кубітом на індексі 0 і застосовуємо до нього гейт Адамара. Це створює стан суперпозиції, який, коли за ним слідує операція керованого-X, генерує стан Белла $(|00\rangle + |11\rangle)/\sqrt{2}$ між керуючим та цільовим кубітами.

На цьому етапі ми ще не конструюємо сам дальній керований-X (LRCX). Натомість наша мета полягає в тому, щоб визначити чітку та мінімальну початкову схему, яка підкреслює роль LRCX. У Кроці 2 ми покажемо, як LRCX може бути реалізований як оптимізація з використанням динамічних схем, і порівняємо його продуктивність з унітарним еквівалентом. Важливо, що протокол LRCX може бути застосований до будь-якої початкової схеми. Тут ми використовуємо це просте налаштування з Адамаром для ясності демонстрації.

In [ ]:
distance = 6  # The distance of the CNOT gate, with the convention that a distance of zero is a nearest-neighbor CNOT.


def initialize_circuit(distance):
    assert distance >= 0
    control = 0  # control qubit
    n = distance  # number of qubits between target and control

    qr = QuantumRegister(
        n + 2, name="q"
    )  # Circuit with n qubits between control and target
    cr = ClassicalRegister(
        2, name="cr"
    )  # Classical register for measuring control and target qubits

    k = int(n / 2)  # Number of Bell States to be used

    allcr = [cr]
    if (
        distance > 1
    ):  # This classical register will be used to store ZZ measurements.
        # It is only used for long-range CX gates with distance > 1
        c1 = ClassicalRegister(
            k, name="c1"
        )  # Classical register needed for post processing
        allcr.append(c1)
    if (
        distance > 0
    ):  # This classical register will be used to store XX measurements.
        # It is only used if distance > 0
        c2 = ClassicalRegister(
            n - k, name="c2"
        )  # Classical register needed for post processing
        allcr.append(c2)

    qc = QuantumCircuit(qr, *allcr, name="CNOT")

    # Apply a Hadamard gate to the control qubit such that the
    # long-range CNOT gate will prepare a
    # Bell state (|00> + |11>)/sqrt(2)
    qc.h(control)

    return qc


qc = initialize_circuit(distance)
qc.draw(fold=-1, output="mpl", scale=0.5)

<Image src="../docs/images/tutorials/long-range-entanglement/extracted-outputs/0446b8e8-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/long-range-entanglement/extracted-outputs/0446b8e8-0.avif)

### Крок 2: Оптимізація задачі для виконання на квантовому обладнанні
На цьому кроці ми показуємо, як побудувати схему LRCX з використанням динамічних схем. Мета полягає в оптимізації схеми для виконання на обладнанні шляхом зменшення глибини порівняно з суто унітарною реалізацією. Щоб проілюструвати переваги, ми покажемо як динамічну конструкцію LRCX, так і її унітарний еквівалент, і пізніше порівняємо їх продуктивність після транспіляції. Важливо, що хоча тут ми застосовуємо LRCX до простої задачі, ініціалізованої Адамаром, протокол може бути застосований до будь-якої схеми, де потрібен дальній CNOT.

#### Підготовка пар Белла
Ми починаємо зі створення ланцюжка пар Белла уздовж шляху між керуючим та цільовим кубітами. Якщо відстань непарна, ми спочатку застосовуємо CNOT від керуючого кубіта до його сусіда, що є CNOT, який буде телепортований. Для парної відстані цей CNOT буде застосовано після кроку підготовки пари Белла. Ланцюжок пар Белла потім заплутує послідовні пари кубітів, встановлюючи ресурс, необхідний для передачі керуючої інформації через пристрій.

In [4]:
# Determine where to start the Bell pair chain and add an extra CNOT when n is odd
def check_even(n: int) -> int:
    """Return 1 if n is even, else 2."""
    return 1 if n % 2 == 0 else 2


def prepare_bell_pairs(qc, add_barriers=True):
    n = qc.num_qubits - 2  # number of qubits between target and control
    k = int(n / 2)

    if add_barriers:
        qc.barrier()

    x0 = check_even(n)
    if n % 2 != 0:
        qc.cx(0, 1)

    # Create k Bell pairs
    for i in range(k):
        qc.h(x0 + 2 * i)
        qc.cx(x0 + 2 * i, x0 + 2 * i + 1)
    return qc


qc = prepare_bell_pairs(qc)
qc.draw(output="mpl", fold=-1, scale=0.5)

<Image src="../docs/images/tutorials/long-range-entanglement/extracted-outputs/4df8ebba-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/long-range-entanglement/extracted-outputs/4df8ebba-0.avif)

#### Вимірювання сусідніх пар кубітів у базисі Белла
Далі ми вимірюємо *незаплутані* сусідні кубіти в базисі Белла (двокубітні вимірювання $XX$ та $ZZ$). Це створює дальню пару Белла між цільовим кубітом та кубітом, сусіднім з керуючим (з точністю до корекцій Паулі, які будуть реалізовані через прямий зв'язок на наступному кроці). Паралельно ми реалізуємо заплутуюче вимірювання, яке телепортує гейт CNOT для дії на цільовий кубіт.

In [ ]:
def measure_bell_basis(qc, add_barriers=True):
    n = qc.num_qubits - 2  # number of qubits between target and control
    k = int(n / 2)

    if n > 1:
        _, c1, c2 = qc.cregs
    elif n > 0:
        _, c2 = qc.cregs

    # Determine where to start the Bell pair chain and add an extra CNOT
    # when n is odd
    x0 = 1 if n % 2 == 0 else 2

    # Entangling layer that implements the Bell measurement
    # (and additionally adds the CNOT to be
    # teleported, if n is even)
    for i in range(k + 1):
        qc.cx(x0 - 1 + 2 * i, x0 + 2 * i)

    for i in range(1, k + x0):
        if i == 1:
            qc.h(2 * i + 1 - x0)
        else:
            qc.h(2 * i + 1 - x0)

    if add_barriers:
        qc.barrier()

    # Map the ZZ measurements onto classical register c1
    for i in range(k):
        if i == 0:
            qc.measure(2 * i + x0, c1[i])
        else:
            qc.measure(2 * i + x0, c1[i])

    # Map the XX measurements onto classical register c2
    for i in range(1, k + x0):
        if i == 1:
            qc.measure(2 * i + 1 - x0, c2[i - 1])
        else:
            qc.measure(2 * i + 1 - x0, c2[i - 1])
    return qc


qc = measure_bell_basis(qc)
qc.draw(output="mpl", fold=-1, scale=0.5)

<Image src="../docs/images/tutorials/long-range-entanglement/extracted-outputs/8eed9e57-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/long-range-entanglement/extracted-outputs/8eed9e57-0.avif)

#### Застосування корекцій прямого зв'язку для виправлення побічних операторів Паулі
Вимірювання в базисі Белла вносять побічні оператори Паулі, які повинні бути виправлені з використанням записаних результатів. Це робиться у два кроки. Спочатку нам потрібно обчислити парність всіх вимірювань $ZZ$, яка потім використовується для умовного застосування гейта $X$ до цільового кубіта. Аналогічно, обчислюється парність вимірювань $XX$ і використовується для умовного застосування гейта $Z$ до керуючого кубіта.

З новою структурою класичних виразів у Qiskit ці парності можуть бути обчислені безпосередньо в шарі класичної обробки схеми. Замість застосування послідовності окремих умовних гейтів для кожного біта вимірювання, ми можемо побудувати єдиний класичний вираз, який представляє XOR (парність) всіх відповідних результатів вимірювання. Цей вираз потім використовується як умова в одному блоці `if_test`, що дозволяє застосовувати корекційні гейти з постійною глибиною. Цей підхід як спрощує схему, так і гарантує, що корекції прямого зв'язку не вносять непотрібної додаткової затримки.

In [6]:
def apply_ffwd_corrections(qc):
    control = 0  # control qubit
    target = qc.num_qubits - 1  # target qubit
    n = qc.num_qubits - 2  # number of qubits between target and control

    k = int(n / 2)
    x0 = check_even(n)

    if n > 1:
        _, c1, c2 = qc.cregs
    elif n > 0:
        _, c2 = qc.cregs

    # First, let's compute the parity of all ZZ measurements
    for i in range(k):
        if i == 0:
            parity_ZZ = expr.lift(
                c1[i]
            )  # Store the value of the first ZZ measurement in parity_ZZ
        else:
            parity_ZZ = expr.bit_xor(
                c1[i], parity_ZZ
            )  # Successively compute the parity via XOR operations

    for i in range(1, k + x0):
        if i == 1:
            parity_XX = expr.lift(
                c2[i - 1]
            )  # Store the value of the first XX measurement in parity_XX
        else:
            parity_XX = expr.bit_xor(
                c2[i - 1], parity_XX
            )  # Successively compute the parity via XOR operations

    if n > 0:
        with qc.if_test(parity_XX):
            qc.z(control)

    if n > 1:
        with qc.if_test(parity_ZZ):
            qc.x(target)
    return qc


qc = apply_ffwd_corrections(qc)
qc.draw(output="mpl", fold=-1, scale=0.5)

<Image src="../docs/images/tutorials/long-range-entanglement/extracted-outputs/4915791a-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/long-range-entanglement/extracted-outputs/4915791a-0.avif)

#### Вимірювання керуючого та цільового кубітів
Ми визначаємо допоміжну функцію, яка дозволяє вимірювати керуючий та цільовий кубіти в базисах $XX$, $YY$ або $ZZ$. Для перевірки стану Белла $(|00\rangle + |11\rangle)/\sqrt{2}$, очікувані значення $XX$ та $ZZ$ повинні обидва дорівнювати $+1$, оскільки вони є стабілізаторами стану. Вимірювання $YY$ також підтримується тут і буде використано нижче при обчисленні точності.

In [ ]:
def measure_in_basis(qc, basis="XX", add_barrier=True):
    control = 0  # control qubit
    target = qc.num_qubits - 1  # target qubit

    assert basis in ["XX", "YY", "ZZ"]

    qc = (
        qc.copy()
    )  # We copy the circuit because we want to measure in different bases
    cr = qc.cregs[0]

    if add_barrier:
        qc.barrier()

    if basis == "XX":
        qc.h(control)
        qc.h(target)
    elif basis == "YY":
        qc.sdg(control)
        qc.sdg(target)
        qc.h(control)
        qc.h(target)

    qc.measure(control, cr[0])
    qc.measure(target, cr[1])
    return qc


qc_YY = measure_in_basis(qc.copy(), basis="YY")
qc_YY.draw(
    output="mpl", fold=-1, scale=0.5
)  # Circuit for measuring in the YY basis

<Image src="../docs/images/tutorials/long-range-entanglement/extracted-outputs/d087d7c1-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/long-range-entanglement/extracted-outputs/d087d7c1-0.avif)

#### Об'єднання всього разом
Ми поєднуємо різні кроки, визначені вище, щоб створити дальній гейт CX на двох кінцях одновимірної (1D) лінії. Кроки включають наступне:
- Ініціалізацію керуючого кубіта в $|+\rangle$
- Підготовку пар Белла
- Вимірювання сусідніх пар кубітів
- Застосування корекцій прямого зв'язку, що залежать від MCM

In [ ]:
def lrcx(distance, prep_barrier=True, pre_measure_barrier=True):
    qc = initialize_circuit(distance)
    qc = prepare_bell_pairs(qc, prep_barrier)
    qc = measure_bell_basis(qc, pre_measure_barrier)
    qc = apply_ffwd_corrections(qc)
    return qc


qc = lrcx(distance)
# Apply the measurement in the XX, YY, and ZZ bases
qc_XX, qc_YY, qc_ZZ = [
    measure_in_basis(qc, basis=basis) for basis in ["XX", "YY", "ZZ"]
]

qc_YY.draw(
    output="mpl", fold=-1, scale=0.5
)  # Circuit for measuring in the YY basis

<Image src="../docs/images/tutorials/long-range-entanglement/extracted-outputs/11fc8adc-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/long-range-entanglement/extracted-outputs/11fc8adc-0.avif)

#### Унітарна реалізація зі свопінгом кубітів до середини
Для порівняння ми спочатку розглянемо випадок, коли довгодіапазонний гейт CNOT реалізується з використанням з'єднань найближчих сусідів та унітарних гейтів. На наступному малюнку зліва показана схема для довгодіапазонного гейта CNOT, що охоплює 1D ланцюг з n-кубітів, обмежений лише з'єднаннями найближчих сусідів. В середині показана еквівалентна унітарна декомпозиція, яку можна реалізувати за допомогою локальних гейтів CNOT, з глибиною схеми $O(n)$.

![Long-range CNOT circuit](../docs/images/tutorials/long-range-entanglement/dynamic_vs_unitary_long_range_illustration.avif)

Схему в середині можна реалізувати наступним чином:

In [ ]:
def cnot_unitary(distance):
    """Generate a long range CNOT gate using local CNOTs on a 1D
    chain of qubits subject to n
    nearest-neighbor connections only.


    Args:
        distance (int) : The distance of the CNOT gate,
        with the convention that
        a distance of 0 is a nearest-neighbor CNOT.

    Returns:
        QuantumCircuit: A Quantum Circuit implementing a
        long-range CNOT gate
        between qubit 0 and qubit distance+1
    """
    assert distance >= 0
    n = distance  # number of qubits between target and control

    qr = QuantumRegister(
        n + 2, name="q"
    )  # Circuit with n qubits between control and target
    cr = ClassicalRegister(
        2, name="cr"
    )  # Classical register for measuring control and target qubits

    qc = QuantumCircuit(qr, cr, name="CNOT_unitary")

    control_qubit = 0

    qc.h(control_qubit)  # Prepare the control qubit in the |+> state

    k = int(n / 2)
    qc.barrier()
    for i in range(control_qubit, control_qubit + k):
        qc.cx(i, i + 1)
        qc.cx(i + 1, i)
        qc.cx(-i - 1, -i - 2)
        qc.cx(-i - 2, -i - 1)
    if n % 2 == 1:
        qc.cx(k + 2, k + 1)
        qc.cx(k + 1, k + 2)
    qc.barrier()
    qc.cx(k, k + 1)
    for i in range(control_qubit, control_qubit + k):
        qc.cx(k - i, k - 1 - i)
        qc.cx(k - 1 - i, k - i)
        qc.cx(k + i + 1, k + i + 2)
        qc.cx(k + i + 2, k + i + 1)
    if n % 2 == 1:
        qc.cx(-2, -1)
        qc.cx(-1, -2)

    return qc


qc_uni = cnot_unitary(distance)

Тепер побудуємо схеми, які вимірюють у базисах $XX$, $YY$ та $ZZ$, точно так само, як ми робили для динамічних схем вище.

In [ ]:
# Apply the measurement in the XX, YY, and ZZ bases
qc_uni_XX, qc_uni_YY, qc_uni_ZZ = [
    measure_in_basis(qc_uni, basis=basis) for basis in ["XX", "YY", "ZZ"]
]

qc_uni_YY.draw(
    output="mpl", fold=-1, scale=0.5
)  # Circuit for measuring in the YY basis

<Image src="../docs/images/tutorials/long-range-entanglement/extracted-outputs/b899e143-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/long-range-entanglement/extracted-outputs/b899e143-0.avif)

Тепер, коли ми побудували як динамічні, так і унітарні схеми для прикладу малого масштабу з `distance=6`, ми транспілюємо їх для запуску спочатку на симуляторі без шуму.

In [10]:
from qiskit_aer import AerSimulator

aer_backend = AerSimulator()
pm_sim = generate_preset_pass_manager(
    optimization_level=0, backend=aer_backend
)

# Dynamic circuits
isa_sim_dyn = pm_sim.run([qc_XX, qc_YY, qc_ZZ])

# Unitary circuits
isa_sim_uni = pm_sim.run([qc_uni_XX, qc_uni_YY, qc_uni_ZZ])

### Крок 3: Виконання з використанням примітивів Qiskit
Тепер ми можемо запустити експеримент на бекенді симулятора без шуму. Ми використовуємо Qiskit Runtime Sampler з AerSimulator як режим бекенду для виконання схем.

In [13]:
sampler_sim = Sampler(mode=aer_backend)
sim_job = sampler_sim.run(isa_sim_dyn + isa_sim_uni)
sim_results = sim_job.result()

### Крок 4: Пост-обробка та повернення результату в бажаному класичному форматі

Після того, як експерименти успішно виконано, ми тепер виконуємо пост-обробку підрахунків вимірювань для отримання значущих метрик.
На цьому кроці ми робимо наступне:

- Визначаємо метрики якості для оцінювання продуктивності довгодіючого CX.
- Обчислюємо очікувані значення операторів Паулі з необроблених результатів вимірювань.
- Використовуємо їх для обчислення точності згенерованого стану Белла.

У симуляції без шуму ми перевіримо, що метрика точності дорівнює $1$ для побудованих схем. В експериментах на реальних QPU цей аналіз надасть чітку картину того, наскільки добре динамічні схеми працюють відносно унітарної базової реалізації.

#### Метрики якості

Щоб оцінити успіх протоколу довгодіючого CX, ми вимірюємо, наскільки близьким вихідний стан є до ідеального стану Белла. Зручним способом кількісно оцінити це є обчислення точності стану за допомогою очікуваних значень операторів Паулі. Ми можемо обчислити точність для стану Белла на керуючому і цільовому стані після знання $\braket{XX}$, $\braket{YY}$ і $\braket{ZZ}$. Зокрема,

$$ F = \frac{1}{4} (1 + \braket{XX} - \braket{YY} + \braket{ZZ})$$

Щоб обчислити ці очікувані значення з необроблених даних вимірювань, ми визначаємо набір допоміжних функцій:

- **`compute_ZZ_expectation`**: За заданими підрахунками вимірювань обчислює очікуване значення двокубітного оператора Паулі в базисі $Z$.
- **`compute_fidelity`**: Об'єднує очікувані значення $XX$, $YY$ і $ZZ$ у вищенаведений вираз точності.
- **`get_counts_from_bitarray`**: Утиліта для витягування підрахунків з об'єктів результатів бекенду.

In [36]:
def compute_ZZ_expectation(counts):
    total = sum(counts.values())
    expectation = 0
    for bitstring, count in counts.items():
        # Ensure bitstring is 2 bits
        z1 = (-1) ** (int(bitstring[-1]))
        z2 = (-1) ** (int(bitstring[-2]))
        expectation += z1 * z2 * count
    return expectation / total


def compute_fidelity(counts_xx, counts_yy, counts_zz):
    xx, yy, zz = [
        compute_ZZ_expectation(c) for c in [counts_xx, counts_yy, counts_zz]
    ]
    return 1 / 4 * (1 + xx - yy + zz)

In [37]:
# Dynamic fidelity
counts_xx = sim_results[0].data.cr.get_counts()
counts_yy = sim_results[1].data.cr.get_counts()
counts_zz = sim_results[2].data.cr.get_counts()
fidelity_dyn = compute_fidelity(counts_xx, counts_yy, counts_zz)

# Unitary fidelity
counts_xx = sim_results[3].data.cr.get_counts()
counts_yy = sim_results[4].data.cr.get_counts()
counts_zz = sim_results[5].data.cr.get_counts()
fidelity_uni = compute_fidelity(counts_xx, counts_yy, counts_zz)

print(f"Dynamic fidelity (distance={distance}): {fidelity_dyn:.4f}")
print(f"Unitary fidelity (distance={distance}): {fidelity_uni:.4f}")

Dynamic fidelity (distance=6): 1.0000
Unitary fidelity (distance=6): 1.0000


As expected in a noiseless simulation, the fidelities in both dynamic circuits and unitary circuits are $1$.

## Large-scale hardware example

Here we now put all of these details together into a single workflow at a larger scale, which is then run on real quantum hardware.

### Generate circuits for different distances

We now generate long-range CX circuits for a range of qubit separations up to 60 qubits apart. For each distance, we build circuits that measure in the $XX$, $YY$, and $ZZ$ bases, which will later be used to compute fidelities.

The list of distances includes both short- and long-range separations, with `distance = 0` corresponding to a nearest-neighbor CX. These same distances will also be used to generate the corresponding unitary circuits later for comparison.

In [ ]:
# -------------------------Step 1-------------------------
distances = [
    0,
    1,
    2,
    3,
    6,
    11,
    16,
    21,
    28,
    35,
    44,
    55,
    60,
]  # Distances for long range CX. distance of 0 is a nearest-neighbor CX
distances.sort()
assert min(distances) >= 0
basis_list = ["XX", "YY", "ZZ"]

# Dynamic circuits
circuits_dyn = []
for distance in distances:
    for basis in basis_list:
        circuits_dyn.append(
            measure_in_basis(lrcx(distance, prep_barrier=False), basis=basis)
        )
print(f"Number of circuits: {len(circuits_dyn)}")

# Unitary circuits
circuits_uni = []
for distance in distances:
    for basis in basis_list:
        circuits_uni.append(
            measure_in_basis(cnot_unitary(distance), basis=basis)
        )

print(f"Number of circuits: {len(circuits_uni)}")

Як і очікувалось у симуляції без шуму, точність як у динамічних, так і в унітарних схемах дорівнює $1$.
## Приклад на реальному обладнанні великого масштабу
Тут ми об'єднуємо всі ці деталі в єдиний робочий процес у більшому масштабі, який потім запускається на реальному квантовому обладнанні.
### Генерація схем для різних відстаней
Тепер ми генеруємо схеми довгодіапазонних CX для діапазону відстаней до 60 кубітів. Для кожної відстані ми будуємо схеми, які вимірюють у базисах $XX$, $YY$ та $ZZ$, які пізніше будуть використані для обчислення точності.

Список відстаней включає як короткодіапазонні, так і довгодіапазонні відстані, де `distance = 0` відповідає найближчому сусідньому CX. Ці ж самі відстані також будуть використані для генерації відповідних унітарних схем пізніше для порівняння.

#### Використання рядка Layer Fidelity для вибору 1D ланцюга
Оскільки ми хочемо порівняти продуктивність динамічних та унітарних схем на 1D ланцюзі, ми використовуємо рядок Layer Fidelity для вибору лінійної топології найкращого ланцюга кубітів з пристрою. Це гарантує, що обидва типи схем транспілюються за однакових обмежень зв'язності, що дозволяє справедливо порівняти їхню продуктивність.

In [24]:
# -------------------------Step 2-------------------------
# Set up access to IBM Quantum devices
from qiskit.circuit import IfElseOp

service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=156
)

The following step ensures that the backend supports the `if_else` instruction, which is required for the newer version of dynamic circuits. Since this feature is still in early access, we explicitly add the `IfElseOp` to the backend target if it is not already available.

In [25]:
if "if_else" not in backend.target.operation_names:
    backend.target.add_instruction(IfElseOp, name="if_else")

#### Use Layer Fidelity string for selecting 1D chain
Since we want to compare the performance of dynamic and unitary circuits on a 1D chain, we use the Layer Fidelity string to select a linear topology of the best chain of qubits from the device. This ensures that both types of circuits are transpiled under the same connectivity constraints, allowing for a fair comparison of their performance.

In [ ]:
# This selects best qubits for longest distance and uses
# the same control for all lengths
lf_qubits = backend.properties().to_dict()[
    "general_qlists"
]  # best linear chain qubits
chosen_layouts = {
    distance: [
        val["qubits"]
        for val in lf_qubits
        if val["name"] == f"lf_{distances[-1] + 2}"
    ][0][: distance + 2]
    for distance in distances
}
print(chosen_layouts[max(distances)])  # best qubits at each distance

[11, 12, 13, 14, 15, 19, 35, 34, 33, 39, 53, 54, 55, 59, 75, 74, 73, 72, 71, 70, 69, 68, 67, 66, 65, 64, 63, 62, 61, 76, 81, 82, 83, 84, 85, 86, 87, 97, 107, 108, 109, 110, 111, 98, 91, 92, 93, 94, 95, 99, 115, 114, 113, 119, 133, 132, 131, 138, 151, 150, 149, 148]


In [ ]:
isa_circuits_dyn = []
isa_circuits_uni = []

# Using the same initial layouts for both circuits for better
# apples to apples comparison
for qc in circuits_dyn:
    pm = generate_preset_pass_manager(
        optimization_level=1,
        backend=backend,
        initial_layout=chosen_layouts[qc.num_qubits - 2],
    )
    isa_circuits_dyn.append(pm.run(qc))

for qc in circuits_uni:
    pm = generate_preset_pass_manager(
        optimization_level=1,
        backend=backend,
        initial_layout=chosen_layouts[qc.num_qubits - 2],
    )
    isa_circuits_uni.append(pm.run(qc))

In [ ]:
print(
    f"2Q depth: "
    f"{isa_circuits_dyn[14].depth(lambda x: x.operation.num_qubits == 2)}"
)
isa_circuits_dyn[14].draw("mpl", fold=-1, idle_wires=0)

2Q depth: 2


<Image src="../docs/images/tutorials/long-range-entanglement/extracted-outputs/c77c3fd3-1.avif" alt="Output of the previous code cell" />

In [ ]:
print(
    f"2Q depth: "
    f"{isa_circuits_uni[14].depth(lambda x: x.operation.num_qubits == 2)}"
)
isa_circuits_uni[14].draw("mpl", fold=-1, idle_wires=False)

2Q depth: 13


<Image src="../docs/images/tutorials/long-range-entanglement/extracted-outputs/7e5fc240-1.avif" alt="Output of the previous code cell" />

### Visualize qubits used for the LRCX circuit

In this section, we examine how the LRCX circuit is mapped onto hardware. We start by visualizing the physical qubits used in the circuit and then study how the control–target distance in the layout impacts the number of operations.

In [ ]:
# Note: the qubit coordinates must be hard-coded.
# The backend API does not currently provide this information directly.
# If using a different backend, you will need to
# adjust the coordinates accordingly,
# or set the qubit_coordinates = None to use the default layout coordinates.


def _heron_coords_r2():
    """Generate coordinates for the Heron layout in R2. Note"""
    cord_map = np.array(
        [
            [
                0,
                1,
                2,
                3,
                4,
                5,
                6,
                7,
                8,
                9,
                10,
                11,
                12,
                13,
                14,
                15,
                3,
                7,
                11,
                15,
                0,
                1,
                2,
                3,
                4,
                5,
                6,
                7,
                8,
                9,
                10,
                11,
                12,
                13,
                14,
                15,
                1,
                5,
                9,
                13,
                0,
                1,
                2,
                3,
                4,
                5,
                6,
                7,
                8,
                9,
                10,
                11,
                12,
                13,
                14,
                15,
                3,
                7,
                11,
                15,
                0,
                1,
                2,
                3,
                4,
                5,
                6,
                7,
                8,
                9,
                10,
                11,
                12,
                13,
                14,
                15,
                1,
                5,
                9,
                13,
                0,
                1,
                2,
                3,
                4,
                5,
                6,
                7,
                8,
                9,
                10,
                11,
                12,
                13,
                14,
                15,
                3,
                7,
                11,
                15,
                0,
                1,
                2,
                3,
                4,
                5,
                6,
                7,
                8,
                9,
                10,
                11,
                12,
                13,
                14,
                15,
                1,
                5,
                9,
                13,
                0,
                1,
                2,
                3,
                4,
                5,
                6,
                7,
                8,
                9,
                10,
                11,
                12,
                13,
                14,
                15,
                3,
                7,
                11,
                15,
                0,
                1,
                2,
                3,
                4,
                5,
                6,
                7,
                8,
                9,
                10,
                11,
                12,
                13,
                14,
                15,
            ],
            -1
            * np.array([j for i in range(15) for j in [i] * [16, 4][i % 2]]),
        ],
        dtype=int,
    )

    hcords = []
    ycords = cord_map[0]
    xcords = cord_map[1]
    for i in range(156):
        hcords.append([xcords[i] + 1, np.abs(ycords[i]) + 1])

    return hcords


# Visualize the active qubits in the circuit layout
plot_circuit_layout(
    circuit=isa_circuits_uni[-1],
    backend=backend,
    view="physical",
    qubit_coordinates=_heron_coords_r2(),
)

<Image src="../docs/images/tutorials/long-range-entanglement/extracted-outputs/2d090f8a-0.avif" alt="Output of the previous code cell" />

Next, we execute the experiment on the real backend. We also make use of batching to efficiently run the experiment across multiple trials. Running repeated trials allows us to compute averages for a more accurate comparison between the unitary and dynamic methods, as well as to quantify their variability by comparing the deviations across runs.

In [31]:
# -------------------------Step 3-------------------------
num_trials = 10
jobs_uni = []
jobs_dyn = []
with Batch(backend=backend) as batch:
    sampler = Sampler(mode=batch)
    sampler.options.environment.job_tags = ["TUT_LRE"]
    for _ in range(num_trials):
        jobs_uni.append(sampler.run(isa_circuits_uni, shots=1024))
        jobs_dyn.append(sampler.run(isa_circuits_dyn, shots=1024))

We compute the fidelity for the dynamic long-range CX circuits. For each distance, we extract measurement outcomes in the $\braket{XX}$, $\braket{YY}$, and $\braket{ZZ}$ bases. These results are combined using the previously defined helper functions to calculate the fidelity according to  $F = \tfrac{1}{4} \big( 1 + \langle XX \rangle - \langle YY \rangle + \langle ZZ \rangle \big)$. This provides the observed fidelity of the dynamically executed protocol at each distance.

In [32]:
# -------------------------Step 4-------------------------
fidelities_dyn = []

# loop over trials
for job in jobs_dyn:
    result_dyn = job.result()
    trial_fidelities = []
    # loop over all distances
    for ind, dist in enumerate(distances):
        counts_xx = result_dyn[ind * 3].data.cr.get_counts()
        counts_yy = result_dyn[ind * 3 + 1].data.cr.get_counts()
        counts_zz = result_dyn[ind * 3 + 2].data.cr.get_counts()
        trial_fidelities.append(
            compute_fidelity(counts_xx, counts_yy, counts_zz)
        )
    fidelities_dyn.append(trial_fidelities)
# average over trials for each distance
avg_fidelities_dyn = np.mean(fidelities_dyn, axis=0)
std_fidelities_dyn = np.std(fidelities_dyn, axis=0)

![Output of the previous code cell](../docs/images/tutorials/long-range-entanglement/extracted-outputs/7e5fc240-1.avif)
### Візуалізація кубітів, використаних для схеми LRCX
У цьому розділі ми розглянемо, як схема LRCX відображається на апаратне забезпечення. Ми почнемо з візуалізації фізичних кубітів, використаних у схемі, а потім дослідимо, як відстань керування-ціль у розміщенні впливає на кількість операцій.

In [33]:
fidelities_uni = []

# loop over trials
for job in jobs_uni:
    result_uni = job.result()
    trial_fidelities = []
    # loop over all distances
    for ind, dist in enumerate(distances):
        counts_xx = result_uni[ind * 3].data.cr.get_counts()
        counts_yy = result_uni[ind * 3 + 1].data.cr.get_counts()
        counts_zz = result_uni[ind * 3 + 2].data.cr.get_counts()
        trial_fidelities.append(
            compute_fidelity(counts_xx, counts_yy, counts_zz)
        )
    fidelities_uni.append(trial_fidelities)
# average over trials for each distance
avg_fidelities_uni = np.mean(fidelities_uni, axis=0)
std_fidelities_uni = np.std(fidelities_uni, axis=0)

![Output of the previous code cell](../docs/images/tutorials/long-range-entanglement/extracted-outputs/2d090f8a-0.avif)

Далі ми виконуємо експеримент на реальному бекенді. Ми також використовуємо пакетування для ефективного виконання експерименту в декількох випробуваннях. Виконання повторних випробувань дозволяє нам обчислити середні значення для більш точного порівняння між унітарним та динамічним методами, а також для кількісної оцінки їх мінливості шляхом порівняння відхилень між запусками.

In [34]:
fig, ax = plt.subplots()

# Unitary with error bars
ax.errorbar(
    distances,
    avg_fidelities_uni,
    yerr=std_fidelities_uni,
    fmt="o-.",
    color="c",
    ecolor="c",
    elinewidth=1,
    capsize=4,
    label="Unitary",
)
# Dynamic with error bars
ax.errorbar(
    distances,
    avg_fidelities_dyn,
    yerr=std_fidelities_dyn,
    fmt="o-.",
    color="m",
    ecolor="m",
    elinewidth=1,
    capsize=4,
    label="Dynamic",
)
# Random gate baseline
ax.axhline(y=1 / 4, linestyle="--", color="gray", label="Random gate")

legend = ax.legend(frameon=True)
for text in legend.get_texts():
    text.set_color("black")
legend.get_frame().set_facecolor("white")
legend.get_frame().set_edgecolor("black")
ax.set_title(
    "Bell State Fidelity vs Control–Target Separation", color="black"
)
ax.set_xlabel("Distance", color="black")
ax.set_ylabel("Bell state fidelity", color="black")
ax.grid(linestyle=":", linewidth=0.6, alpha=0.4, color="gray")
ax.set_ylim((0.2, 1))
ax.set_facecolor("white")
fig.patch.set_facecolor("white")
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color("black")
ax.tick_params(axis="x", colors="black")
ax.tick_params(axis="y", colors="black")
plt.show()

<Image src="../docs/images/tutorials/long-range-entanglement/extracted-outputs/724da22d-0.avif" alt="Output of the previous code cell" />

From the fidelity plot above, the LRCX did not consistently outperform the direct unitary implementation. In fact, for short control–target separations, the unitary circuit achieved higher fidelity. However, at larger separations, the dynamic circuit begins to achieve better fidelity than the unitary implementation. This behavior is not unexpected on current hardware: while dynamic circuits reduce circuit depth by avoiding long SWAP chains, they introduce additional circuit time from mid-circuit measurements, classical feedforward, and control-path delays. The added latency increases decoherence and readout errors, which can outweigh the depth savings at short distances.

Nevertheless, we observe a crossover point where the dynamic approach surpasses the unitary one. This is a direct result of the different scaling: the depth of the unitary circuit grows linearly with the distance between qubits, while the depth of the dynamic circuit remains constant.

**Key points:**
- **Immediate benefit of dynamic circuits:** The main present-day motivation is reduced *two-qubit depth*, not necessarily improved fidelity.
- **Why fidelity can be worse today:** Increased circuit time from measurement and classical operations often dominates, especially when the control–target separation is small.
- **Looking forward:** As hardware improves, specifically faster readout, shorter classical control latency, and reduced mid-circuit overhead, we should expect these depth and duration reductions to translate into measurable fidelity gains.

In [ ]:
# Compute metrics for each distance, skipping the basis circuits since
# they are identical for each distance
depths_2q_dyn = [
    c.depth(lambda x: x.operation.num_qubits == 2)
    for c in isa_circuits_dyn[::3]
]
meas_dyn = [
    sum(1 for instr in c.data if instr.operation.name == "measure")
    for c in isa_circuits_dyn[::3]
]

depths_2q_uni = [
    c.depth(lambda x: x.operation.num_qubits == 2)
    for c in isa_circuits_uni[::3]
]
meas_uni = [
    sum(1 for instr in c.data if instr.operation.name == "measure")
    for c in isa_circuits_uni[::3]
]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(
    distances, depths_2q_uni, "o-.", color="c", label="Unitary (2Q depth)"
)
axes[0].plot(
    distances, depths_2q_dyn, "o-.", color="m", label="Dynamic (2Q depth)"
)
axes[0].set_xlabel("Number of qubits between control and target")
axes[0].set_ylabel("Two-qubit depth")
axes[0].grid(True, linestyle=":", linewidth=0.6, alpha=0.4)
axes[0].legend()

axes[1].plot(
    distances, meas_uni, "o-.", color="c", label="Unitary (# measurements)"
)
axes[1].plot(
    distances, meas_dyn, "o-.", color="m", label="Dynamic (# measurements)"
)
axes[1].set_xlabel("Number of qubits between control and target")
axes[1].set_ylabel("Number of measurements")
axes[1].grid(True, linestyle=":", linewidth=0.6, alpha=0.4)
axes[1].legend()

fig.suptitle("Scaling of Unitary vs Dynamic LRCX with Distance", fontsize=12)

plt.tight_layout()
plt.show()

<Image src="../docs/images/tutorials/long-range-entanglement/extracted-outputs/3dcff343-0.avif" alt="Output of the previous code cell" />

Ми обчислюємо точність для динамічних схем довгодіючого CX. Для кожної відстані ми витягуємо результати вимірювань у базисах $\braket{XX}$, $\braket{YY}$ і $\braket{ZZ}$. Ці результати об'єднуються за допомогою раніше визначених допоміжних функцій для обчислення точності згідно з $F = \tfrac{1}{4} \big( 1 + \langle XX \rangle - \langle YY \rangle + \langle ZZ \rangle \big)$. Це надає спостережувану точність протоколу з динамічним виконанням на кожній відстані.